# Setup Data in Local Repository
Run this code to configure the data folder in personal repository. 
For this code to work, follow these steps

* Download fma_small.zip and fma_metadata.zip from [the README.md in this repository](https://github.com/mdeff/fma/tree/master) 
* Put the downloaded zip files in your local cloned repository (data/fma_small.zip and data/fma_metadata.zip)
* Extract fma_metadata.zip in the same folder (data/fma_metadata/)
* Make sure to add "data/" to the .gitignore file so that the data is not committed to github


Imports and Paths

In [1]:
import zipfile
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
ZIP_PATH = DATA_DIR / "fma_small.zip"

Count Songs in ZIP

In [2]:
with zipfile.ZipFile(ZIP_PATH) as z:
    names = [n for n in z.namelist() if n.endswith(".mp3")]

print(len(names), "songs")
print(names[:3])

8000 songs
['fma_small/000/000002.mp3', 'fma_small/000/000005.mp3', 'fma_small/000/000010.mp3']


Build a table with track ID as the key

In [3]:
songs = pd.DataFrame({"zip_path": names})
songs["track_id"] = songs["zip_path"].apply(lambda p: int(Path(p).stem))
songs = songs.set_index("track_id").sort_index()
songs.head()

,zip_path
track_id,
2,fma_small/000/000002.mp3
5,fma_small/000/000005.mp3
10,fma_small/000/000010.mp3
140,fma_small/000/000140.mp3
141,fma_small/000/000141.mp3


Extract each song

In [4]:
EXTRACT_DIR = DATA_DIR / "fma_small_mp3"

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)

songs["path"] = songs["zip_path"].apply(lambda p: EXTRACT_DIR / p)

Load the metadata

In [5]:
tracks = pd.read_csv(
    DATA_DIR / "fma_metadata" / "tracks.csv",   # after unzipping fma_metadata.zip
    index_col=0,
    header=[0, 1],   # tracks.csv has a two-row header
)

small = tracks[tracks["set", "subset"] == "small"]
print(small.shape)   # should be (8000, ...)

small[[("track", "title"), ("artist", "name"), ("track", "genre_top")]].head()

(8000, 52)


,track,artist,track
,title,name,genre_top
track_id,,,
2,Food,AWOL,Hip-Hop
5,This World,AWOL,Hip-Hop
10,Freeway,Kurt Vile,Pop
140,Queen Of The Wires,Alec K. Redfearn & the Eyesores,Folk
141,Ohio,Alec K. Redfearn & the Eyesores,Folk


Join with the tracks table

In [7]:
cols = {
    ("track", "title"): "title",
    ("artist", "name"): "artist",
    ("track", "genre_top"): "genre",
}

small_meta = small[list(cols.keys())].copy()
small_meta.columns = list(cols.values())

meta = songs.join(small_meta)
meta.head()

,zip_path,path,title,artist,genre
track_id,,,,,
2,fma_small/000/000002.mp3,../data/fma_small_mp3/fma_small/000/000002.mp3,Food,AWOL,Hip-Hop
5,fma_small/000/000005.mp3,../data/fma_small_mp3/fma_small/000/000005.mp3,This World,AWOL,Hip-Hop
10,fma_small/000/000010.mp3,../data/fma_small_mp3/fma_small/000/000010.mp3,Freeway,Kurt Vile,Pop
140,fma_small/000/000140.mp3,../data/fma_small_mp3/fma_small/000/000140.mp3,Queen Of The Wires,Alec K. Redfearn & the Eyesores,Folk
141,fma_small/000/000141.mp3,../data/fma_small_mp3/fma_small/000/000141.mp3,Ohio,Alec K. Redfearn & the Eyesores,Folk
